# 📊🤖 Mesa Financiera en Telegram — Ruta A (Google Colab)
**Dra. Elda C. Morales · Actividad para principiantes**

Construirás un ecosistema de **3 agentes** que trabajan en cadena, dirigidos por **Batuta**, el orquestador:

| Agente | Recibe | Hace | Entrega |
|---|---|---|---|
| 🛰️ **Iris · La Recolectora** (Agente 1) | Ticker que escribes en Telegram | Consulta datos reales de Alpha Vantage mediante **MCP** | Ficha JSON |
| 🧮 **Euler · El Analista** (Agente 2) | La ficha de Iris | Python **calcula** indicadores; Claude los **interpreta** | Análisis JSON |
| ✍️ **Vera · La Comunicadora** (Agente 3) | El análisis de Euler | Redacta una nota clara | Mensaje en Telegram |

**Cómo usar este notebook:** ejecuta las celdas en orden con ▶ (o `Shift + Enter`). Después de cada celda revisa el apartado *"Deberías ver"*.

> 💡 ¿Aún no tienes llaves? Deja `MODO_SIMULACION = True` en la celda 2: todo funciona con datos ficticios.

## Celda 1 · Instalar librerías
**Deberías ver:** la celda termina sin errores en rojo (puede tardar unos 20 segundos).

In [ ]:
!pip install -q anthropic requests

## Celda 2 · Configuración y llaves
Las llaves **nunca** se escriben en el código. Guárdalas en el panel **🔑 Secretos** (barra izquierda de Colab) → *Add new secret*, con estos nombres exactos y el interruptor **Notebook access** encendido:

`TELEGRAM_BOT_TOKEN` · `TELEGRAM_CHAT_ID` · `ANTHROPIC_API_KEY` · `ALPHAVANTAGE_API_KEY`

**Deberías ver:** una tabla con ✅ en cada llave configurada y el modo (SIMULACIÓN o REAL).

In [ ]:
import json, os, re, time
import requests

MODO_SIMULACION = True        # ← cámbialo a False cuando tengas tus 4 llaves
MODO_CLASE      = True        # True = el bot muestra cada entrega entre agentes
MODELO          = "claude-haiku-5-5"

def leer_secreto(nombre):
    """Lee un secreto de Colab (o de una variable de entorno fuera de Colab)."""
    try:
        from google.colab import userdata
        return (userdata.get(nombre) or "").strip()
    except Exception:
        return os.environ.get(nombre, "").strip()

TELEGRAM_TOKEN   = leer_secreto("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = leer_secreto("TELEGRAM_CHAT_ID")
ANTHROPIC_KEY    = leer_secreto("ANTHROPIC_API_KEY")
ALPHAVANTAGE_KEY = leer_secreto("ALPHAVANTAGE_API_KEY")

if not MODO_SIMULACION and not (ANTHROPIC_KEY and ALPHAVANTAGE_KEY):
    print("⚠️ Faltan ANTHROPIC_API_KEY o ALPHAVANTAGE_API_KEY: activo el MODO SIMULACIÓN.")
    MODO_SIMULACION = True

for nombre, valor in [("TELEGRAM_BOT_TOKEN", TELEGRAM_TOKEN), ("TELEGRAM_CHAT_ID", TELEGRAM_CHAT_ID),
                      ("ANTHROPIC_API_KEY", ANTHROPIC_KEY), ("ALPHAVANTAGE_API_KEY", ALPHAVANTAGE_KEY)]:
    print(f"{nombre:<22} {'✅ configurada' if valor else '⬜ vacía'}")   # nunca se imprime el valor
print("Modo:", "SIMULACIÓN (datos ficticios)" if MODO_SIMULACION else "REAL")

## Celda 3 · Conexión con Claude y con el servidor MCP
Aquí ocurre la integración **MCP**: con solo indicar la dirección del servidor de Alpha Vantage, Claude puede usar sus herramientas de datos bursátiles. No se instala nada más.

**Deberías ver:** `✅ Conexión preparada`.

In [ ]:
DESCARGO = "Nota educativa, no es recomendación de inversión."
cliente = None
if not MODO_SIMULACION:
    import anthropic
    cliente = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

def llamar_claude(system, mensaje, usar_mcp=False, max_tokens=900):
    """Envía un mensaje a Claude. Devuelve (texto, herramientas_MCP_usadas)."""
    base = dict(model=MODELO, max_tokens=max_tokens, system=system,
                messages=[{"role": "user", "content": mensaje}])
    if usar_mcp:
        resp = cliente.beta.messages.create(
            **base,
            mcp_servers=[{"type": "url", "name": "alphavantage",
                          "url": f"https://mcp.alphavantage.co/mcp?apikey={ALPHAVANTAGE_KEY}"}],
            tools=[{"type": "mcp_toolset", "mcp_server_name": "alphavantage"}],
            betas=["mcp-client-2025-11-20"])
    else:
        resp = cliente.messages.create(**base)
    texto = "".join(b.text for b in resp.content if b.type == "text").strip()
    return texto, [b.name for b in resp.content if b.type == "mcp_tool_use"]

def extraer_json(texto):
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin <= inicio:
        raise ValueError("El agente no devolvió JSON")
    return json.loads(texto[inicio:fin + 1])

def a_numero(valor):
    try:
        return float(str(valor).replace("%", "").replace(",", ""))
    except (TypeError, ValueError):
        return None

print("✅ Conexión preparada")

## Celda 4 · 🛰️ Iris, la Recolectora (Agente 1)
Curiosa y meticulosa: pide a Claude que consulte Alpha Vantage por MCP y devuelva una **ficha JSON**. Si el MCP falla o se agota la cuota diaria, usa datos DEMO para que la clase no se detenga.

**Deberías ver:** `✅ Iris lista`.

In [ ]:
PROMPT_RECOLECTOR = """Eres Iris, la Recolectora (Agente 1) de una mesa financiera educativa.
Tu personalidad: curiosa y meticulosa; jamás inventas un dato.
Tarea: obtener datos REALES de la acción indicada con las herramientas de Alpha Vantage.
Reglas:
- Haz como máximo 2 consultas: cotización actual y datos generales de la empresa.
- NUNCA inventes cifras. Si un dato no está disponible, escribe null.
- Responde SOLO con un JSON con estas claves exactas:
  ticker, nombre, sector, precio, cambio_pct, fecha_dato, pe_ratio, beta,
  max_52s, min_52s, cap_mercado_usd"""

DATOS_DEMO = {"nombre": "Empresa ficticia (simulación)", "sector": "Tecnología", "precio": 150.0,
              "cambio_pct": 1.2, "fecha_dato": "dato ficticio", "pe_ratio": 28.0, "beta": 1.25,
              "max_52s": 180.0, "min_52s": 120.0, "cap_mercado_usd": None}

def agente_recolector(ticker):
    if MODO_SIMULACION:
        return {"ticker": ticker, **DATOS_DEMO, "fuente": "DATOS DEMO, no son reales (modo simulación)"}, []
    try:
        texto, herramientas = llamar_claude(PROMPT_RECOLECTOR, f"Acción: {ticker}", usar_mcp=True)
        ficha = extraer_json(texto)
        ficha.update(ticker=ticker, fuente="Alpha Vantage vía MCP")
        return ficha, herramientas
    except Exception as e:
        print("⚠️ MCP no disponible, uso datos DEMO:", type(e).__name__)
        return {"ticker": ticker, **DATOS_DEMO, "fuente": "DATOS DEMO, no son reales (MCP no disponible)"}, []

print("✅ Iris lista")

## Celda 5 · 🧮 Euler, el Analista (Agente 2)
**Regla de oro:** los números los calcula el código; la IA solo interpreta.

- Posición en el rango de 52 semanas: $P_{52} = \dfrac{P_t - P_{\min}}{P_{\max} - P_{\min}} \times 100$
- Semáforo de beta (umbrales didácticos): 🟢 β < 0.8 · 🟡 0.8 ≤ β ≤ 1.2 · 🔴 β > 1.2

**Deberías ver:** `{'posicion_rango_52s_pct': 50.5, ...}` y `✅ Euler listo`.

In [ ]:
def calcular_indicadores(ficha):
    precio, maxi, mini = (a_numero(ficha.get(k)) for k in ("precio", "max_52s", "min_52s"))
    beta = a_numero(ficha.get("beta"))
    posicion = None
    if None not in (precio, maxi, mini) and maxi > mini:
        posicion = round((precio - mini) / (maxi - mini) * 100, 1)
    if beta is None:   semaforo = "⚪ sin dato de beta"
    elif beta < 0.8:   semaforo = "🟢 volatilidad baja"
    elif beta <= 1.2:  semaforo = "🟡 volatilidad similar al mercado"
    else:              semaforo = "🔴 volatilidad alta"
    return {"posicion_rango_52s_pct": posicion, "semaforo_beta": semaforo, "beta": beta,
            "pe_ratio": a_numero(ficha.get("pe_ratio")), "cambio_pct": a_numero(ficha.get("cambio_pct"))}

PROMPT_ANALISTA = """Eres Euler, el Analista (Agente 2). Recibes la ficha de Iris y unos indicadores ya calculados.
Tu personalidad: riguroso con los números y paciente para explicarlos.
NO recalcules ni cambies cifras. Interprétalas para un principiante.
Responde SOLO con JSON con las claves:
  lectura_precio (1 frase), lectura_riesgo (1 frase sobre beta),
  lectura_valuacion (1 frase sobre P/E, aclarando que depende del sector),
  pregunta_para_investigar (1 pregunta que el alumno debería investigar)"""

def agente_analista(ficha):
    ind = calcular_indicadores(ficha)
    if MODO_SIMULACION:
        pos = ind["posicion_rango_52s_pct"]
        lectura = {"lectura_precio": f"El precio está al {pos} % de su rango de 52 semanas.",
                   "lectura_riesgo": f"Semáforo de beta: {ind['semaforo_beta']}.",
                   "lectura_valuacion": "El P/E solo se interpreta comparándolo con su sector.",
                   "pregunta_para_investigar": "¿Cuál es el P/E promedio del sector de esta empresa?"}
    else:
        texto, _ = llamar_claude(PROMPT_ANALISTA, json.dumps({"ficha": ficha, "indicadores": ind}, ensure_ascii=False))
        try:
            lectura = extraer_json(texto)
        except ValueError:
            lectura = {"lectura_precio": texto[:300]}
    return {"ficha": ficha, "indicadores": ind, "lectura": lectura}

print(calcular_indicadores({"precio": 250.5, "max_52s": 300, "min_52s": 200, "beta": 0.7}))  # comprobación
print("✅ Euler listo")

## Celda 6 · ✍️ Vera, la Comunicadora (Agente 3)
Clara, breve y honesta. **Deberías ver:** `✅ Vera lista`.

In [ ]:
PROMPT_COMUNICADOR = f"""Eres Vera, la Comunicadora (Agente 3). Con el análisis de Euler, redacta una nota para Telegram.
Tu personalidad: clara, breve y honesta sobre los límites del análisis.
- Máximo 900 caracteres, español claro, 3 a 5 emojis como máximo.
- Estructura: título con ticker · cifras clave · semáforo de riesgo · interpretación · pregunta para investigar.
- Usa solo las cifras recibidas; si la fuente dice DEMO, adviértelo en la primera línea.
- Cierra SIEMPRE con: "{DESCARGO}"
Texto plano, sin Markdown."""

def agente_comunicador(analisis):
    f, ind, lec = analisis["ficha"], analisis["indicadores"], analisis["lectura"]
    if MODO_SIMULACION:
        return (f"✍️ Vera · Comunicadora\n⚠️ {f['fuente']}\n📊 {f['ticker']} · {f['nombre']}\n"
                f"Precio: {f['precio']} USD ({ind['cambio_pct']} % en el día)\n"
                f"Riesgo: {ind['semaforo_beta']} (beta {ind['beta']})\n{lec['lectura_precio']}\n"
                f"🔎 {lec['pregunta_para_investigar']}\n\n{DESCARGO}")
    texto, _ = llamar_claude(PROMPT_COMUNICADOR, json.dumps(analisis, ensure_ascii=False), max_tokens=600)
    texto = texto if DESCARGO in texto else f"{texto}\n\n{DESCARGO}"
    return f"✍️ Vera · Comunicadora\n{texto}"

print("✅ Vera lista")

## Celda 7 · 🎼 Batuta, el orquestador, y una prueba sin Telegram
Batuta decide el orden y **pasa el output de cada agente como input del siguiente**.

**Deberías ver:** las entregas `1→2` y `2→3` y, al final, la nota con el descargo de responsabilidad.
En modo real, la ficha debe decir `"fuente": "Alpha Vantage vía MCP"`. Toma **captura** de esta salida.

In [ ]:
def pipeline(ticker, avisar=print):
    ticker = ticker.strip().upper()
    if not re.fullmatch(r"[A-Z0-9.\-]{1,12}", ticker):
        raise ValueError(f"Ticker no válido: {ticker}")
    avisar(f"🎼 Batuta · Orquestador\nRecibí tu orden. Inicio la cadena para {ticker}.")
    avisar(f"🛰️ Iris · Recolectora buscando datos de {ticker}…")
    ficha, herramientas = agente_recolector(ticker)
    if MODO_CLASE:
        avisar("📦 Entrega 1→2 · Iris → Euler (ficha):\n" + json.dumps(ficha, ensure_ascii=False, indent=1)[:1500]
               + f"\n🔧 Herramientas MCP usadas: {', '.join(herramientas) or 'ninguna'}")
    avisar("🧮 Euler · Analista calculando e interpretando…")
    analisis = agente_analista(ficha)
    if MODO_CLASE:
        avisar("📦 Entrega 2→3 · Euler → Vera (indicadores):\n" + json.dumps(analisis["indicadores"], ensure_ascii=False, indent=1))
    avisar("✍️ Vera · Comunicadora redactando la nota…")
    return agente_comunicador(analisis)

print(pipeline("IBM"))

## Celda 8 · 📱 Telegram como centro de mando
Al ejecutar esta celda el bot queda **escuchando**. Escríbele desde Telegram: `/analiza AAPL`.

- Para apagarlo, pulsa ■ (detener) en la celda.
- Si cierras la pestaña, el bot se apaga. Para que funcione con la computadora apagada usa la **Ruta B (GitHub Actions)**.

**Deberías ver:** `✅ Bot @tu_bot escuchando` y, en Telegram, los mensajes de Batuta, Iris, Euler y Vera.

In [ ]:
API_TG = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
AYUDA = ("🤖 Mesa Financiera de 3 agentes\n/analiza TICKER → ejemplo: /analiza AAPL\n"
         "Batuta dirige: Iris (MCP) → Euler → Vera")

def enviar(chat_id, texto):
    requests.post(f"{API_TG}/sendMessage", data={"chat_id": chat_id, "text": texto[:4000]}, timeout=20)

def atender(mensaje):
    chat_id, texto = mensaje["chat"]["id"], (mensaje.get("text") or "").strip()
    if TELEGRAM_CHAT_ID and str(chat_id) != TELEGRAM_CHAT_ID:
        return enviar(chat_id, "Este bot es privado.")
    if not texto.lower().startswith("/analiza"):
        return enviar(chat_id, AYUDA)
    partes = texto.split()
    try:
        nota = pipeline(partes[1] if len(partes) > 1 else "", avisar=lambda t: enviar(chat_id, t))
        enviar(chat_id, nota)
    except ValueError:
        enviar(chat_id, "Escribe un ticker válido. Ejemplo: /analiza MSFT")
    except Exception as e:
        enviar(chat_id, f"⚠️ No pude completar el análisis ({type(e).__name__}). Revisa llaves y saldo.")

def encender_bot():
    yo = requests.get(f"{API_TG}/getMe", timeout=20).json()
    if not yo.get("ok"):
        raise RuntimeError("❌ Token de Telegram inválido o vacío. Revísalo con @BotFather.")
    requests.get(f"{API_TG}/deleteWebhook", timeout=20)
    print(f"✅ Bot @{yo['result']['username']} escuchando. Pulsa ■ para apagarlo.")
    offset = None
    try:
        while True:
            try:
                r = requests.get(f"{API_TG}/getUpdates", params={"timeout": 30, "offset": offset}, timeout=45).json()
                for upd in r.get("result", []):
                    offset = upd["update_id"] + 1
                    if "message" in upd:
                        atender(upd["message"])
            except requests.RequestException as e:
                print("Conexión inestable, reintento en 5 s:", e)
                time.sleep(5)
    except KeyboardInterrupt:
        print("👋 Bot detenido.")

if TELEGRAM_TOKEN:
    encender_bot()
else:
    print("⬜ Falta TELEGRAM_BOT_TOKEN: crea tu bot con @BotFather y guárdalo en 🔑 Secretos.")

## ✅ Lista de validación
- [ ] La celda 7 muestra las entregas 1→2 y 2→3.
- [ ] En modo real, la fuente dice **Alpha Vantage vía MCP** y aparecen herramientas usadas.
- [ ] En Telegram, `/analiza AAPL` devuelve tres avisos y una nota final con el descargo.
- [ ] `/analiza AAPL;x` responde "Escribe un ticker válido".

> Material educativo de la Dra. Elda C. Morales. Las notas **no son recomendaciones de inversión**.